# Lab | Data Aggregation and Filtering

In this challenge, we will continue to work with customer data from an insurance company. We will use the dataset called marketing_customer_analysis.csv, which can be found at the following link:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis.csv

This dataset contains information such as customer demographics, policy details, vehicle information, and the customer's response to the last marketing campaign. Our goal is to explore and analyze this data by first performing data cleaning, formatting, and structuring.

1. Create a new DataFrame that only includes customers who:
   - have a **low total_claim_amount** (e.g., below $1,000),
   - have a response "Yes" to the last marketing campaign.

2. Using the original Dataframe, analyze:
   - the average `monthly_premium` and/or customer lifetime value by `policy_type` and `gender` for customers who responded "Yes", and
   - compare these insights to `total_claim_amount` patterns, and discuss which segments appear most profitable or low-risk for the company.

3. Analyze the total number of customers who have policies in each state, and then filter the results to only include states where there are more than 500 customers.

4. Find the maximum, minimum, and median customer lifetime value by education level and gender. Write your conclusions.

## Bonus

5. The marketing team wants to analyze the number of policies sold by state and month. Present the data in a table where the months are arranged as columns and the states are arranged as rows.

6.  Display a new DataFrame that contains the number of policies sold by month, by state, for the top 3 states with the highest number of policies sold.

*Hint:*
- *To accomplish this, you will first need to group the data by state and month, then count the number of policies sold for each group. Afterwards, you will need to sort the data by the count of policies sold in descending order.*
- *Next, you will select the top 3 states with the highest number of policies sold.*
- *Finally, you will create a new DataFrame that contains the number of policies sold by month for each of the top 3 states.*

7. The marketing team wants to analyze the effect of different marketing channels on the customer response rate.

Hint: You can use melt to unpivot the data and create a table that shows the customer response rate (those who responded "Yes") by marketing channel.

External Resources for Data Filtering: https://towardsdatascience.com/filtering-data-frames-in-pandas-b570b1f834b9

In [1]:
import pandas as pd

url = "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis.csv"
df = pd.read_csv(url)
print(df.head())

   Unnamed: 0 Customer       State  Customer Lifetime Value Response  \
0           0  DK49336     Arizona              4809.216960       No   
1           1  KX64629  California              2228.525238       No   
2           2  LZ68649  Washington             14947.917300       No   
3           3  XL78013      Oregon             22332.439460      Yes   
4           4  QA50777      Oregon              9025.067525       No   

   Coverage Education Effective To Date EmploymentStatus Gender  ...  \
0     Basic   College           2/18/11         Employed      M  ...   
1     Basic   College           1/18/11       Unemployed      F  ...   
2     Basic  Bachelor           2/10/11         Employed      M  ...   
3  Extended   College           1/11/11         Employed      M  ...   
4   Premium  Bachelor           1/17/11    Medical Leave      F  ...   

   Number of Open Complaints Number of Policies     Policy Type        Policy  \
0                        0.0                  9  Corp

In [7]:
df = pd.read_csv(url)

In [8]:
df.columns = df.columns.str.lower().str.replace(' ', '_')

In [9]:
filtered_df = df[
    (df['total_claim_amount'] < 1000) & 
    (df['response'].str.strip().str.capitalize() == 'Yes')
].copy()


print(f"Número de clientes filtrados: {len(filtered_df)}")
print(filtered_df[['customer', 'response', 'total_claim_amount']].head())

Número de clientes filtrados: 1399
   customer response  total_claim_amount
3   XL78013      Yes          484.013411
8   FM55990      Yes          739.200000
15  CW49887      Yes          547.200000
19  NJ54277      Yes           19.575683
27  MQ68407      Yes           60.036683


In [11]:
filtered_df = df[
    (df['total_claim_amount'] < 1000) & 
    (df['response'].str.strip().str.capitalize() == 'Yes')
].copy()

In [12]:
print(f"Número de clientes filtrados: {len(filtered_df)}")
print(filtered_df[['customer', 'response', 'total_claim_amount']].head())

Número de clientes filtrados: 1399
   customer response  total_claim_amount
3   XL78013      Yes          484.013411
8   FM55990      Yes          739.200000
15  CW49887      Yes          547.200000
19  NJ54277      Yes           19.575683
27  MQ68407      Yes           60.036683


In [ ]:
if 'monthly_premium_auto' in df.columns:
    df.rename(columns={'monthly_premium_auto': 'monthly_premium'}, inplace=True)


df_responded = df[df['response'].str.strip().str.capitalize() == 'Yes'].copy()


analysis_segmentation = df_responded.groupby(['policy_type', 'gender']).agg(
    avg_monthly_premium=('monthly_premium', 'mean'),
    avg_clv=('customer_lifetime_value', 'mean'),
    avg_total_claim_amount=('total_claim_amount', 'mean'),
    customer_count=('customer', 'count')
).reset_index()


analysis_segmentation['estimated_annual_premium'] = analysis_segmentation['avg_monthly_premium'] * 12
analysis_segmentation['claim_to_annual_premium_ratio'] = (
    analysis_segmentation['avg_total_claim_amount'] / analysis_segmentation['estimated_annual_premium']
)

print(analysis_segmentation)

      policy_type gender  avg_monthly_premium      avg_clv  \
0  Corporate Auto      F            94.301775  7712.628736   
1  Corporate Auto      M            92.188312  7944.465414   
2   Personal Auto      F            98.998148  8339.791842   
3   Personal Auto      M            91.085821  7448.383281   
4    Special Auto      F            92.314286  7691.584111   
5    Special Auto      M            86.343750  8247.088702   

   avg_total_claim_amount  customer_count  estimated_annual_premium  \
0              433.738499             169               1131.621302   
1              408.582459             154               1106.259740   
2              452.965929             540               1187.977778   
3              457.010178             536               1093.029851   
4              453.280164              35               1107.771429   
5              429.527942              32               1036.125000   

   claim_to_annual_premium_ratio  
0                       0.383289

In [14]:
customers_per_state = (
    df.groupby('state')['customer']
    .count()
    .reset_index(name='customer_count')
)

In [15]:
states_above_500 = customers_per_state[customers_per_state['customer_count'] > 500].sort_values(
    by='customer_count', ascending=False
)

In [16]:
print("Total customers per state (Filtered > 500):")
print(states_above_500)

Total customers per state (Filtered > 500):
        state  customer_count
1  California            3552
3      Oregon            2909
0     Arizona            1937
2      Nevada             993
4  Washington             888


In [19]:
clv_by_education_gender = (
    df.groupby(['education', 'gender'])['customer_lifetime_value']
    .agg(
        max_clv='max',
        min_clv='min',
        median_clv='median',
        customer_count='count'
    )
    .reset_index()
)

clv_by_education_gender['max_clv'] = clv_by_education_gender['max_clv'].round(2)
clv_by_education_gender['min_clv'] = clv_by_education_gender['min_clv'].round(2)
clv_by_education_gender['median_clv'] = clv_by_education_gender['median_clv'].round(2)

print(clv_by_education_gender)

              education gender   max_clv  min_clv  median_clv  customer_count
0              Bachelor      F  73225.96  1904.00     5640.51            1700
1              Bachelor      M  67907.27  1898.01     5548.03            1572
2               College      F  61850.19  1898.68     5623.61            1608
3               College      M  61134.68  1918.12     6005.85            1596
4                Doctor      F  44856.11  2395.57     5332.46             200
5                Doctor      M  32677.34  2267.60     5577.67             201
6  High School or Below      F  55277.45  2144.92     6039.55            1590
7  High School or Below      M  83325.38  1940.98     6286.73            1556
8                Master      F  51016.07  2417.78     5729.86             475
9                Master      M  50568.26  2272.31     5579.10             412
